[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C55_TSR_Autonomous_Driving_Course/02_pipeline/02_tsr_pipeline_design.ipynb)

# 02 · TSR 系统设计：两级 vs 端到端（级联误差 / 延迟 p99 / crop 契约 / 置信度标定）

目标：把「两级方案好不好」这个含糊的架构问题，变成**四个可以算出数字的问题**——
级联召回怎么乘、延迟的 p99 怎么控、crop 契约错了会丢多少点、置信度怎么才算「真概率」。

本 notebook 你会亲手实现：

1. **级联误差传播计算器** + 边际收益/成本的改进优先级排序（∂R/∂p 的实际用法）
2. **两级 vs 端到端的延迟模型**：crop 数随场景波动 → p50/p99 → 固定 batch 如何用均值换掉方差
3. **长尾下的精度模拟**：为什么两级赢在尾部而不是头部
4. **crop 契约实验**：在 GT crop 上训练、在检测 crop 上推理，凭空损失多少准确率（本模块最重要的实验）
5. **ECE + 可靠性图 + temperature scaling**：把两个分数变成一个可信的概率
6. **代价敏感的拒识阈值**：τ = c_FP / (c_FP + c_FN)

> 心智模型：**级联系统的性能在对数域上是相加的，没有任何一级能补偿另一级的损失。
> 所以「前松后紧」，并且永远只汇报端到端。**

## 1 · 级联误差传播计算器

`R_end2end = R_det × A_cls`。先把这个乘法的后果算出来。

In [ ]:
import numpy as np, math, json
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

def cascade_recall(rates):
    '''级联通过率 = 各级通过率之积。整个模块的核心公式。'''
    r = 1.0
    for p in rates:
        r *= p
    return r

CASES = [
    ('两级 · 各 0.90',        [0.90, 0.90]),
    ('两级 · 汇报值 0.96/0.97', [0.96, 0.97]),
    ('两级 · 真实条件准确率',   [0.96, 0.92]),
    ('三级 · 各 0.95',        [0.95, 0.95, 0.95]),
    ('五级 · 各 0.98',        [0.98] * 5),
]
print(f"{'配置':<22s}{'各级通过率':<28s}{'端到端':>9s}{'最弱一级':>10s}")
for name, r in CASES:
    print(f"{name:<22s}{str(r):<28s}{cascade_recall(r):>9.3f}{min(r):>10.3f}")

assert abs(cascade_recall([0.9, 0.9]) - 0.81) < 1e-12
assert abs(cascade_recall([0.98] * 5) - 0.98 ** 5) < 1e-12
assert cascade_recall([0.96, 0.92]) < 0.90, '两个 90+ 的模块合起来掉到 90 以下'

print()
print('⚠️  0.90 × 0.90 = 0.81 —— 两个「还行」的模块合起来是「不能用」。')
print('⚠️  0.96 × 0.97 = 0.931，而不是 PM 记下的「96%+」。')
print('✅ 对外只能汇报端到端；分级指标永远系统性高估。')
print('✅ 每加一级就多一次乘法 —— 五级各 0.98 也只剩 %.3f。' % cascade_recall([0.98] * 5))

In [ ]:
def others_product(rates, i):
    '''除第 i 级之外所有级的乘积 —— 它就是第 i 级的「边际收益系数」。'''
    p = 1.0
    for j, v in enumerate(rates):
        if j != i:
            p *= v
    return p

def marginal_gain(rates, i, delta):
    '''把第 i 级提升 delta 个绝对点，端到端涨多少。'''
    new = list(rates)
    new[i] = min(1.0, new[i] + delta)
    return cascade_recall(new) - cascade_recall(rates)

rates = [0.92, 0.94]                      # [R_det, A_cls]
names = ['检测召回 R_det', '分类准确率 A_cls']
print(f'当前端到端 = {cascade_recall(rates):.4f}')
for i in range(2):
    g = marginal_gain(rates, i, 0.03)
    print(f'  {names[i]:<18s} +0.03  ->  端到端 +{g:.4f}   ( = 0.03 × {others_product(rates, i):.3f} )')

# 严格恒等式：∂R/∂p_i = 其余各级之积  =>  ΔR = Δp_i × others_product
for i in range(2):
    assert abs(marginal_gain(rates, i, 0.03) - 0.03 * others_product(rates, i)) < 1e-12

print()
print('✅ 两级情况下「其余各级之积」就是另一级的通过率 —— 两者接近时，')
print('   同样 +0.03 的收益几乎相同。**所以光看收益无法决策，必须引入成本与天花板。**')

In [ ]:
# 真实决策：收益 / 成本 / 天花板 三者一起排序
base = cascade_recall(rates)
PLAN = [
    # (方案, 作用于第几级, 天花板, 成本(人月), 线上算力代价)
    ('提高检测输入分辨率', 0, 0.970, 2.0, '延迟 +3.0 ms'),
    ('分类器补长尾数据',   1, 0.985, 1.0, '延迟 +0.0 ms'),
]
print(f"{'方案':<20s}{'可涨到':>8s}{'端到端增量':>12s}{'每人月收益':>12s}{'线上代价':>14s}")
rec = {}
for name, i, ceil_, cost, lat in PLAN:
    r2 = list(rates); r2[i] = ceil_
    gain = cascade_recall(r2) - base
    rec[name] = (gain, gain / cost)
    print(f'{name:<20s}{ceil_:>8.3f}{gain:>12.4f}{gain / cost:>12.4f}{lat:>14s}')

g_det, roi_det = rec['提高检测输入分辨率']
g_cls, roi_cls = rec['分类器补长尾数据']
assert g_det > g_cls, '按绝对增量：检测的空间更大'
assert roi_cls > roi_det, '按每人月收益：补分类数据更划算'

print()
print('⚠️  按「绝对增量」排 -> 先做检测；按「每人月收益」排 -> 先做分类。**排序会反转。**')
print('✅ 车端还要再乘一个约束：检测方案要吃 3 ms 延迟，而分类方案是 0。')
print('   ⇒ 算力受限时，同等收益优先改分类。这就是模块讲解里那条结论的来源。')

## 2 · 两级 vs 端到端：延迟模型与 p99

车端真正在意的不是均值，是 **p99**。而两级方案的第二级延迟正比于 crop 数，
**crop 数随场景剧烈波动**（空旷路段 2 个，复杂路口 30 个）—— 这是方差的来源。

In [ ]:
T_DET, T_CLS, T_OVH, T_POST = 7.0, 0.22, 0.4, 1.0    # ms：检测/单 crop 分类/batch 开销/后处理
T_E2E = 9.6                                            # 端到端多类检测器（无第二级）

def lat_dynamic(n, cap=None):
    '''动态 batch：延迟正比于实际 crop 数（cap 为 top-k 截断上限）。'''
    k = n if cap is None else min(n, cap)
    return T_DET + T_OVH + T_CLS * k + T_POST

def lat_fixed(n, batch=12):
    '''固定 batch：不足补 padding，超出截断 -> 延迟恒定。'''
    return T_DET + T_OVH + T_CLS * batch + T_POST

# 场景混合：55% 空旷 / 35% 城区 / 10% 复杂路口
n_frames = 40000
scene = rng.choice([0, 1, 2], size=n_frames, p=[0.55, 0.35, 0.10])
n_crops = rng.poisson(np.array([2.0, 8.0, 25.0])[scene])

variants = {
    'A 动态 batch，无上限': np.array([lat_dynamic(n) for n in n_crops]),
    'B 动态 + top-12 截断': np.array([lat_dynamic(n, cap=12) for n in n_crops]),
    'C 固定 batch = 12':   np.array([lat_fixed(n) for n in n_crops]),
    'D 端到端（无第二级）':  np.full(n_frames, T_E2E),
}
print(f"{'方案':<22s}{'均值':>8s}{'p50':>8s}{'p99':>8s}{'max':>8s}{'标准差':>9s}")
S = {}
for k, v in variants.items():
    S[k] = (v.mean(), np.percentile(v, 50), np.percentile(v, 99), v.max(), v.std())
    print(f'{k:<22s}' + ''.join(f'{x:>8.2f}' for x in S[k][:4]) + f'{S[k][4]:>9.2f}')

assert S['A 动态 batch，无上限'][2] > S['B 动态 + top-12 截断'][2], 'top-k 截断压住了 p99'
assert S['C 固定 batch = 12'][4] < 1e-12, '固定 batch 的延迟方差应为 0'
assert S['A 动态 batch，无上限'][0] < S['C 固定 batch = 12'][0], '固定 batch 用均值换方差'
assert S['B 动态 + top-12 截断'][2] <= S['C 固定 batch = 12'][2] + 1e-9

print()
print('✅ 车端偏好 C：**用更高的均值换掉方差**，因为调度器按最坏情况分配时间片。')
print('⚠️  A 的均值最低（%.2f ms）却最危险：p99 是 %.2f ms，复杂路口会掉帧。'
      % (S['A 动态 batch，无上限'][0], S['A 动态 batch，无上限'][2]))
print('✅ 两级相比端到端多付 %.2f ms（固定 batch），换来 200+ 类细粒度识别 + 加类不重训。'
      % (S['C 固定 batch = 12'][0] - T_E2E))

In [ ]:
# 截断的代价：top-k 会丢掉真标志吗？
def truncation_recall(k, trials=6000, seed=3):
    '''真标志的检测分数偏高（Beta(6,2)），误检偏低（Beta(2,6)）。'''
    g = np.random.default_rng(seed)
    tot, kept = 0, 0.0
    for _ in range(trials):
        n_true = int(g.integers(1, 4))
        n_false = int(g.poisson(8))
        s = np.concatenate([g.beta(6, 2, size=n_true), g.beta(2, 6, size=n_false)])
        lab = np.concatenate([np.ones(n_true), np.zeros(n_false)])
        idx = np.argsort(-s)[:k]
        tot += n_true
        kept += lab[idx].sum()
    return kept / tot

print(f"{'top-k 上限':>10s}{'真标志召回':>12s}{'分类阶段延迟':>14s}")
recs = {}
for k in [2, 4, 8, 12, 20, 40]:
    r = truncation_recall(k)
    recs[k] = r
    print(f'{k:>10d}{r:>12.4f}{T_OVH + T_CLS * k:>13.2f} ms')

assert recs[12] > recs[4] > recs[2], 'k 越大召回越高'
assert recs[12] > 0.99, 'k=12 已经几乎无损 —— 因为真标志的分数偏高'
assert recs[40] - recs[12] < 0.01, '再往上是纯粹的延迟浪费'

print()
print('✅ 关键洞察：**截断的代价取决于真标志分数是否偏高**。')
print('   在 TSR 里真标志的检测分数系统性高于广告牌误检 -> top-12 几乎无损。')
print('⚠️  但这依赖检测分数「排序正确」。如果检测器过度自信地把广告牌排到前面，')
print('   截断就会开始吃真标志 —— 这又回到了第 5 节的标定问题。')

## 3 · 长尾下的精度模拟：两级赢在尾部，不在头部

用一条学习曲线 `acc(n) = ceiling · (1 − e^(−n/n₀))` 建模「样本量 → 精度」，
两级的两个优势体现为：**检测器的 n 是全类之和**，**分类器的任务更简单（n₀ 更小）且可重采样**。

In [ ]:
def learn_curve(n, n0=350.0, ceiling=0.985):
    '''样本量 -> 精度的经验学习曲线。n0 越小表示任务越「样本高效」。'''
    return ceiling * (1.0 - np.exp(-np.asarray(n, float) / n0))

K_CLS = 120                                  # 细类数
ranks = np.arange(1, K_CLS + 1)
w = ranks ** -1.4                            # Zipf 型长尾
counts = np.maximum(8, np.round(w / w.sum() * 120000)).astype(int)
N_total = counts.sum()
print(f'类别数 {K_CLS} | 实例总数 {N_total} | 最多的类 {counts.max()} | 最少的类 {counts.min()}'
      f' | 头尾比 {counts.max() / counts.min():.0f}:1')

# ── 端到端：每一类都要从自己的 counts[c] 个实例里同时学「定位 + 识别」
acc_e2e = learn_curve(counts, n0=350.0)

# ── 两级：
#   检测器只学「牌状物」，正样本 = 全部类之和
R_det = float(learn_curve(N_total, n0=350.0))
#   分类器任务更简单（输入已归一化，不用学定位）-> n0 更小；且可重采样（LVIS 式 repeat factor）
f = counts / N_total
repeat = np.clip(np.sqrt(0.001 / f), 1.0, 4.0)          # 上限 4，避免过拟合
acc_cls = learn_curve(counts * repeat, n0=110.0)
r_two = R_det * acc_cls

head, tail = slice(0, 10), slice(K_CLS - 30, K_CLS)
print(f'\n检测器召回 R_det = {R_det:.4f}（全类样本共享的好处）')
print(f"{'':<10s}{'端到端 macro':>14s}{'两级 macro':>14s}{'差':>9s}")
for nm, sl in [('全部类', slice(None)), ('头部 10 类', head), ('尾部 30 类', tail)]:
    a, b = acc_e2e[sl].mean(), r_two[sl].mean()
    print(f'{nm:<10s}{a:>14.4f}{b:>14.4f}{b - a:>+9.4f}')

d_head = r_two[head].mean() - acc_e2e[head].mean()
d_tail = r_two[tail].mean() - acc_e2e[tail].mean()
assert r_two.mean() > acc_e2e.mean(), '两级的 macro 更高'
assert abs(d_head) < 0.02, '头部类几乎没差别 —— 数据都够，谁都学得会'
assert d_tail > 0.15, '尾部类差距巨大 —— 这才是两级的真正价值'
assert d_tail > 5 * abs(d_head)

print()
print('✅ **两级的收益几乎全部来自尾部**。头部类样本充足，两种架构都学得会。')
print('⚠️  所以用 micro / 实例加权指标去评估两级 vs 端到端，会**看不见**这个收益')
print('   （尾部类实例少，加权后被淹没）。**必须看 macro 与逐类召回。**')
print('⚠️  注意这是一个「模型」不是「测量」：learn_curve 与 repeat 上限都是假设。')
print('   它的价值是给出**趋势与量级**，用来做决策，而不是预测具体数字。')

## 4 · crop 契约：GT crop 上训练、检测 crop 上推理 = 隐形的域差

**本模块最重要的实验。** 我们造一个可控的合成 TSR：
类别 = 圆形外圈（各类相同）+ 类别专属的内部低频图案；
分类器 = 最近邻 exemplar bank（在裁好的 crop 特征上做匹配）。

然后对比两种训练方式：**用 GT 框裁剪** vs **用带扰动的框裁剪**（模拟检测器输出）。

In [ ]:
K = 8            # 合成细类数
R = 32           # 模板与分类器输入分辨率

def resize(img, out=32):
    '''双线性缩放（纯 numpy）。'''
    H, W = img.shape
    ys = np.linspace(0, H - 1, out); xs = np.linspace(0, W - 1, out)
    y0 = np.floor(ys).astype(int); x0 = np.floor(xs).astype(int)
    y1 = np.minimum(y0 + 1, H - 1); x1 = np.minimum(x0 + 1, W - 1)
    wy = (ys - y0)[:, None]; wx = (xs - x0)[None, :]
    top = img[y0][:, x0] * (1 - wx) + img[y0][:, x1] * wx
    bot = img[y1][:, x0] * (1 - wx) + img[y1][:, x1] * wx
    return top * (1 - wy) + bot * wy

def make_templates(K=8, R=32, seed=1):
    '''外圈（所有类共有，像红边）+ 内部 4×4 低频类别图案。'''
    ax = np.arange(R); yy, xx = np.meshgrid(ax, ax, indexing='ij')
    c = (R - 1) / 2
    r = np.sqrt((yy - c) ** 2 + (xx - c) ** 2) / (R / 2)
    ring = ((r > 0.80) & (r <= 1.0)).astype(float)
    inner = (r <= 0.72).astype(float)
    g = np.random.default_rng(seed)
    return np.stack([ring * 1.5 + inner * np.kron(g.normal(size=(4, 4)),
                                                  np.ones((R // 4, R // 4)))
                     for _ in range(K)])

TPL = make_templates(K, R)

def render(k, s, canvas=120, noise=0.35, g=None):
    '''把第 k 类模板缩放到 s×s 贴到带噪背景上，返回 (图, GT框)。'''
    g = g if g is not None else rng
    img = g.normal(0, noise, size=(canvas, canvas))
    y0 = int(g.integers(10, canvas - s - 10)); x0 = int(g.integers(10, canvas - s - 10))
    img[y0:y0 + s, x0:x0 + s] += resize(TPL[k], s)
    return img, (float(x0), float(y0), float(s), float(s))

def perturb(box, j, g):
    '''模拟检测器输出：中心偏移 ±j·边长，尺度抖动 ±j。'''
    x, y, w, h = box
    cx, cy = x + w / 2 + g.uniform(-j, j) * w, y + h / 2 + g.uniform(-j, j) * h
    sc = 1 + g.uniform(-j, j)
    return (cx - w * sc / 2, cy - h * sc / 2, w * sc, h * sc)

def crop_box(img, box, pad=0.15):
    '''正方形化 + padding + 边界裁剪 —— 这就是两级方案的 API。'''
    x, y, w, h = box
    s = max(w, h); cx, cy = x + w / 2, y + h / 2
    x0 = int(round(cx - s / 2 - pad * s)); y0 = int(round(cy - s / 2 - pad * s))
    x1 = int(round(cx + s / 2 + pad * s)); y1 = int(round(cy + s / 2 + pad * s))
    H, W = img.shape
    x0, y0 = max(0, x0), max(0, y0); x1, y1 = min(W, x1), min(H, y1)
    if x1 <= x0 + 2 or y1 <= y0 + 2:
        return np.zeros((4, 4))
    return img[y0:y1, x0:x1]

def feat(crop, out=32):
    '''归一化的 crop 特征（去均值 + L2 归一）—— 相关系数即余弦相似度。'''
    z = resize(crop, out).ravel()
    z = z - z.mean()
    return z / (np.linalg.norm(z) + 1e-9)

img0, box0 = render(3, 24, g=np.random.default_rng(5))
print('场景', img0.shape, '| GT 框', box0, '| crop', crop_box(img0, box0).shape,
      '| 特征维度', feat(crop_box(img0, box0)).shape)
assert abs(np.linalg.norm(feat(crop_box(img0, box0))) - 1.0) < 1e-6
print('✅ 合成 TSR 就位：8 类、圆形外圈共有、内部图案区分类别。')

In [ ]:
def build_bank(jitter, n_per=40, seed=7):
    '''构建 exemplar bank（= 用某种裁剪方式「训练」分类器）。'''
    g = np.random.default_rng(seed)
    X, Y = [], []
    for k in range(K):
        for _ in range(n_per):
            img, box = render(k, s=int(g.integers(18, 34)), g=g)
            b = perturb(box, jitter, g) if jitter > 0 else box
            X.append(feat(crop_box(img, b))); Y.append(k)
    return np.stack(X), np.array(Y)

def evaluate(bank, jitter, n=400, seed=99):
    X, Y = bank
    g = np.random.default_rng(seed); ok = 0
    for _ in range(n):
        k = int(g.integers(0, K))
        img, box = render(k, s=int(g.integers(18, 34)), g=g)
        b = perturb(box, jitter, g) if jitter > 0 else box
        ok += int(Y[int(np.argmax(X @ feat(crop_box(img, b))))] == k)
    return ok / n

bank_gt  = build_bank(0.00)      # ❌ 用 GT 框裁剪训练（最常见的做法）
bank_jit = build_bank(0.18)      # ✅ 用带扰动的框裁剪训练（模拟检测器输出）

print(f"{'推理时的框扰动':>14s}{'GT-crop 训练':>16s}{'扰动-crop 训练':>16s}{'差':>8s}")
res = {}
for J in [0.00, 0.10, 0.18, 0.25]:
    a, b = evaluate(bank_gt, J), evaluate(bank_jit, J)
    res[J] = (a, b)
    tag = '   <- 离线评测看到的' if J == 0 else ('  <- 线上真实工况' if J == 0.18 else '')
    print(f'{J:>14.2f}{a:>16.3f}{b:>16.3f}{b - a:>+8.3f}{tag}')

assert res[0.00][0] > 0.98, '在 GT crop 上评测，一切完美'
assert res[0.18][0] < res[0.00][0] - 0.25, '换成检测框 crop，准确率断崖式下跌'
assert res[0.18][1] > res[0.18][0] + 0.08, '训练时模拟检测框扰动，能把大部分损失赚回来'

print()
print('⚠️  第一行 vs 第三行就是那个「离线 97%、路测像 91%」的故事：')
print('    在 GT crop 上评测 %.3f -> 真实工况 %.3f，**离线评测完全看不出来**。'
      % (res[0.00][0], res[0.18][0]))
print('✅ 修法只有一条：**分类器的训练数据必须用「模拟检测器输出的框」裁剪**')
print('   （给 GT 框加上与检测器误差分布一致的中心偏移 + 尺度抖动）。')
print('✅ 同样重要：**分类器的验证集也必须用检测框裁**，否则你根本测不到这个问题。')

In [ ]:
# padding 该给多少？用**几何量**来算，比跑分类器更干净：
#   ① 完整包住率 = 带误差的框 + padding 后，crop 仍然完整包住真实牌面的比例（越高越好）
#   ② 牌面占比   = 牌在 crop 中的面积份额（padding 越大越小 —— 这是代价）
def containment_rate(pad, jitter, n=200000, seed=4):
    '''归一化到 GT 边长 = 1：中心偏移 ~ U(-j, j)，尺度 ~ 1 + U(-j, j)。'''
    g = np.random.default_rng(seed)
    dx = g.uniform(-jitter, jitter, n); dy = g.uniform(-jitter, jitter, n)
    half = (1 + g.uniform(-jitter, jitter, n)) * (1 + 2 * pad) / 2
    return float(np.mean((np.abs(dx) + 0.5 <= half) & (np.abs(dy) + 0.5 <= half)))

def sign_occupancy(pad, jitter, n=200000, seed=4):
    g = np.random.default_rng(seed)
    half = (1 + g.uniform(-jitter, jitter, n)) * (1 + 2 * pad) / 2
    return float(np.mean(1.0 / (2 * half) ** 2))

print(f"{'padding':>8s}" + ''.join(f'{f"包住率 j={j:.2f}":>16s}' for j in [0.05, 0.12, 0.20])
      + f"{'牌面占比 j=0.12':>16s}")
C = {}
for pad in [0.00, 0.05, 0.10, 0.15, 0.20, 0.35]:
    C[pad] = [containment_rate(pad, j) for j in [0.05, 0.12, 0.20]]
    print(f'{pad:>8.2f}' + ''.join(f'{v:>16.3f}' for v in C[pad])
          + f'{sign_occupancy(pad, 0.12):>16.3f}')

assert C[0.00][0] < 0.30, 'padding=0：只要框有一点误差，牌面（含最有判别力的外圈）就被切掉'
assert C[0.20][1] > C[0.10][1] > C[0.00][1], '完整包住率随 padding 单调上升'
assert C[0.35][2] > 0.95, '框误差越大，需要的 padding 越大'
assert sign_occupancy(0.35, 0.12) < sign_occupancy(0.10, 0.12), 'padding 的代价：牌在 crop 里变小'

need = min(p for p in C if C[p][1] > 0.98)
print(f'\n在 j=0.12 的框误差下，要 98% 完整包住，至少需要 padding = {need:.2f}')
print('✅ padding 的本质是**吸收检测框误差的余量**：0% 时框一偏，外圈就被切掉 —— 而')
print('   外圈（红边 / 形状）恰恰是 TSR 最有判别力的部分。')
print('✅ 自适应：pad = max(0.15, 3px / min(w,h))。小框的**相对**误差更大，要给更多余量；')
print('   同时 padding 越大牌面占比越小，所以不能无脑加 —— 这是一个可以算出来的取舍。')
print('⚠️  padding 一旦定下来就是**破坏性接口**：改它 = 两个模型都要重训重验。')

## 5 · 置信度合成与标定：ECE、可靠性图、temperature scaling

`s_det × s_cls` 只有在两者**都是标定过的概率**时才有意义。
现实是：检测分数被 focal loss 压低、分类分数系统性过度自信。

In [ ]:
def logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def ece(conf, correct, n_bins=10, return_bins=False):
    '''Expected Calibration Error：按置信度分桶，|准确率 − 平均置信度| 的加权平均。'''
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=False), 0, n_bins - 1)
    e, rows = 0.0, []
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            rows.append((edges[b], edges[b + 1], 0, float('nan'), float('nan')))
            continue
        acc, cf = correct[m].mean(), conf[m].mean()
        e += m.sum() / len(conf) * abs(acc - cf)
        rows.append((edges[b], edges[b + 1], int(m.sum()), cf, acc))
    return (e, rows) if return_bins else e

def reliability(conf, correct, n_bins=10, title=''):
    '''文本版可靠性图：★=实际准确率，│=完美标定应在的位置。'''
    e, rows = ece(conf, correct, n_bins, return_bins=True)
    print(f'{title}  ECE = {e:.4f}')
    print(f"{'区间':<12s}{'样本':>7s}{'平均置信':>9s}{'实际准确':>9s}   {'差':>7s}  图")
    for lo, hi, n, cf, acc in rows:
        if n == 0:
            continue
        bar = [' '] * 42
        bar[min(41, int(cf * 41))] = '|'
        bar[min(41, int(acc * 41))] = '*'
        print(f'[{lo:.1f},{hi:.1f}){n:>9d}{cf:>9.3f}{acc:>9.3f}{acc - cf:>+8.3f}  ' + ''.join(bar))
    return e

n = 30000
p_true = rng.beta(3.0, 1.2, size=n)                 # 每个样本「真正的」正确概率
correct = (rng.random(n) < p_true).astype(float)
conf_raw = sigmoid(logit(p_true) * 2.0)             # 模型报出的分数：logit 被放大 -> 过度自信

e_oracle = ece(p_true, correct)
e_raw = reliability(conf_raw, correct, title='【未标定】高分桶里 * 在 | 左边 = 过度自信')
assert e_oracle < 0.02, '真概率本身几乎完美标定（验证 ECE 实现正确）'
assert e_raw > 0.08, '过度自信的分数 ECE 明显偏大'
assert e_raw > 8 * e_oracle
print()
print('⚠️  看**高置信度那几个桶**：★ 落在 │ 左边 = 实际准确率低于宣称置信度 = 过度自信。')
print('    这是现代深网络的典型症状，在长尾尾部类上更严重。')
print('⚠️  但注意低置信度桶里方向是**反的**（★ 在 │ 右边 = 偏保守）——')
print('    因为「logit 被整体放大」会把 p<0.5 推低、把 p>0.5 推高。')
print('    ⇒ **这两半会在单个 ECE 数字里互相抵消**，所以永远不能只看 ECE。')

In [ ]:
# Temperature scaling：单参数、不改变排序（不影响准确率/mAP）、纯后处理
def nll(conf, correct):
    c = np.clip(conf, 1e-6, 1 - 1e-6)
    return float(-(correct * np.log(c) + (1 - correct) * np.log(1 - c)).mean())

def fit_temperature(conf, correct, grid=np.linspace(0.4, 3.0, 261)):
    '''在**独立的标定集**上最小化 NLL 求 T。'''
    losses = [nll(sigmoid(logit(conf) / T), correct) for T in grid]
    return float(grid[int(np.argmin(losses))])

cal, tst = slice(0, 12000), slice(12000, n)          # 标定集 / 测试集必须分开
T_star = fit_temperature(conf_raw[cal], correct[cal])
conf_cal = sigmoid(logit(conf_raw) / T_star)

e_before = ece(conf_raw[tst], correct[tst])
e_after = ece(conf_cal[tst], correct[tst])
print(f'拟合出的温度 T = {T_star:.3f}   （数据生成时用的是 2.0 -> 应当被恢复出来）')
print(f'测试集 ECE:  标定前 {e_before:.4f}  ->  标定后 {e_after:.4f}   ({e_after / e_before:.1%})')

assert abs(T_star - 2.0) < 0.15, 'temperature scaling 应恢复出真实的过度自信系数'
assert e_after < e_before / 2, '标定后 ECE 至少减半'
order = np.argsort(conf_raw)
assert np.all(np.diff(conf_cal[order]) >= -1e-12), \
    'T>0 是单调变换 -> **排序不变 -> 准确率/mAP 完全不变**'

_ = reliability(conf_cal[tst], correct[tst], title='\n【标定后】')
print()
print('✅ temperature scaling 的三个工程优点：①单参数②不改排序（准确率/mAP 不动）③纯后处理。')
print('⚠️  两个陷阱：① **不能在训练集上标定**（模型在训练集上近乎完美，T≈1，等于没做）；')
print('   ② **标定依赖分布** —— 晴天标出的 T，夜间/雨天依然过度自信 -> 应分场景各标一个 T。')
print('⚠️  别只报一个 ECE 数字：它会让「低分区偏保守」和「高分区过自信」互相抵消。')
print('   **必须看可靠性图，尤其是最高的那两三个桶** —— 下游只用高置信度的检测。')

In [ ]:
# 合成：两个未标定分数相乘 vs 各自标定后相乘
m = 30000
pd_true = rng.beta(3.0, 1.2, size=m)     # 真·「这里有个牌」的概率
pc_true = rng.beta(2.5, 1.2, size=m)     # 真·「类别判对」的条件概率
det_ok = rng.random(m) < pd_true
cls_ok = rng.random(m) < pc_true
joint = (det_ok & cls_ok).astype(float)  # 端到端正确 = 两级都对

sd_raw = sigmoid(logit(pd_true) * 1.5)   # 检测分数：过度自信
sc_raw = sigmoid(logit(pc_true) * 1.9)   # 分类分数：更过度自信

c2 = slice(0, 12000); t2 = slice(12000, m)
Td = fit_temperature(sd_raw[c2], det_ok[c2].astype(float))
Tc = fit_temperature(sc_raw[c2], cls_ok[c2].astype(float))
sd_cal = sigmoid(logit(sd_raw) / Td)
sc_cal = sigmoid(logit(sc_raw) / Tc)

rows = [
    ('未标定相乘  s_det × s_cls', sd_raw[t2] * sc_raw[t2]),
    ('各自标定后相乘',            sd_cal[t2] * sc_cal[t2]),
    ('理论上界（用真概率相乘）',   pd_true[t2] * pc_true[t2]),
    ('只用 s_det（忽略分类）',    sd_raw[t2]),
]
print(f"{'合成方式':<26s}{'ECE(端到端正确)':>16s}")
E = {}
for nm, v in rows:
    E[nm] = ece(v, joint[t2])
    print(f'{nm:<26s}{E[nm]:>16.4f}')

assert E['理论上界（用真概率相乘）'] < 0.02, '链式法则成立：真概率相乘就是端到端的真概率'
assert E['各自标定后相乘'] < E['未标定相乘  s_det × s_cls'] / 2, '分别标定后再相乘，ECE 大幅下降'
assert E['只用 s_det（忽略分类）'] > E['各自标定后相乘'], '丢掉分类置信度会严重高估'
print(f'\n拟合温度：T_det = {Td:.2f}, T_cls = {Tc:.2f}（分类器更过度自信，符合预期）')
print()
print('✅ **乘法只在两者都标定后才有概率语义**。链式法则要求的是概率，不是「分数」。')
print('⚠️  最后一行：只把检测分数传给下游 = 假装分类永远是对的 -> 系统性高估。')
print('   **感知→决策接口最常见的设计错误就是不传或传错置信度**（C59 模块 03 会再讲一次）。')

## 6 · 拒识：阈值按「代价」定，而不是按「准确率」定

softmax 在任何输入上都会给一个答案。拒识就是给它加一道闸门 ——
而闸门的高度应该由 **误报代价 vs 漏检代价** 决定：`τ = c_FP / (c_FP + c_FN)`。

In [ ]:
COST = {                       # (误报代价, 漏检代价) —— 数量级差异是真实的
    '停车让行 STOP': (2.0, 200.0),
    '限速 60':      (8.0, 40.0),
    '禁止掉头':      (15.0, 25.0),
    '景点指示':      (25.0, 1.0),
}

def opt_tau(c_fp, c_fn):
    '''最小期望代价的决策阈值：上报 iff (1−p)·c_FP < p·c_FN。'''
    return c_fp / (c_fp + c_fn)

def realized_cost(p, y, c_fp, c_fn, tau):
    '''p: 标定后的置信度; y: 是否真的是该类; tau: 上报阈值。'''
    report = p > tau
    return float(((report & (y == 0)) * c_fp + ((~report) & (y == 1)) * c_fn).sum())

g = np.random.default_rng(21)
N = 60000
p_hat = g.beta(2.0, 2.0, size=N)
y = (g.random(N) < p_hat).astype(int)

print(f"{'类别':<16s}{'c_FP':>7s}{'c_FN':>7s}{'最优 τ':>9s}{'τ=0.5 代价':>12s}{'最优 τ 代价':>13s}{'省':>8s}")
tot_fixed = tot_opt = 0.0
for name, (cfp, cfn) in COST.items():
    t = opt_tau(cfp, cfn)
    c_fixed = realized_cost(p_hat, y, cfp, cfn, 0.5)
    c_opt = realized_cost(p_hat, y, cfp, cfn, t)
    tot_fixed += c_fixed; tot_opt += c_opt
    print(f'{name:<16s}{cfp:>7.1f}{cfn:>7.1f}{t:>9.3f}{c_fixed:>12.0f}{c_opt:>13.0f}'
          f'{1 - c_opt / c_fixed:>7.1%}')
    assert c_opt <= c_fixed + 1e-9, f'{name}: 代价最优阈值不应比 0.5 差'

assert opt_tau(2.0, 200.0) < 0.05, '漏检代价极高 -> 阈值极低 -> 宁滥勿缺'
assert opt_tau(25.0, 1.0) > 0.90, '误报代价高、漏检无所谓 -> 阈值极高 -> 宁缺勿滥'
assert tot_opt < tot_fixed * 0.85, '逐类阈值显著优于全局 0.5'

print(f'\n总代价：全局 τ=0.5 -> {tot_fixed:.0f}  |  逐类最优 τ -> {tot_opt:.0f}'
      f'  （降低 {1 - tot_opt / tot_fixed:.1%}）')
print()
print('✅ 「停车让行」阈值 %.3f（几乎不拒），「景点指示」阈值 %.3f（几乎全拒）——'
      % (opt_tau(*COST['停车让行 STOP']), opt_tau(*COST['景点指示'])))
print('   **同一个模型，不同类别用完全不同的工作点**，这就是代价敏感决策。')
print('⚠️  前提：p 必须是**标定过的概率**。未标定的分数代进这个不等式，阈值全是错的。')
print('   ⇒ 第 5 节的标定不是「锦上添花」，它是代价敏感决策的**前置条件**。')

## ✏️ 练习 1：改进优先级排序器

实现 `improvement_plan(rates, ceilings, costs)`：

- `rates`：各级当前通过率；`ceilings`：各级能提升到的天花板；`costs`：各级提升到天花板的成本
- 返回 `{'base':…, 'gains':[…], 'roi':[…], 'best_gain':idx, 'best_roi':idx}`
- `gains[i]` = 只把第 i 级提到天花板时的**端到端**增量；`roi[i] = gains[i] / costs[i]`

**这是「下个季度做什么」的决策器**，也是级联公式最直接的工程用途。

In [ ]:
def improvement_plan(rates, ceilings, costs):
    # TODO: ① base = cascade_recall(rates)
    #       ② 对每个 i：把 rates[i] 换成 ceilings[i]，算端到端增量
    #       ③ roi = gain / cost；返回按 gain 与按 roi 的最优下标
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
pl = improvement_plan([0.92, 0.94], [0.970, 0.985], [2.0, 1.0])
assert abs(pl['base'] - 0.8648) < 1e-9
assert abs(pl['gains'][0] - (0.97 * 0.94 - 0.8648)) < 1e-9
assert abs(pl['gains'][1] - (0.92 * 0.985 - 0.8648)) < 1e-9
assert pl['best_gain'] == 0, '按绝对增量：检测'
assert pl['best_roi'] == 1, '按每单位成本收益：分类'
pl3 = improvement_plan([0.90, 0.95, 0.98], [0.96, 0.97, 0.99], [1.0, 1.0, 1.0])
assert pl3['best_gain'] == 0 and pl3['best_roi'] == 0, '三级时最弱的一级空间最大'
assert abs(sum(pl3['gains']) - sum(pl3['roi'])) < 1e-9, '成本全为 1 时 gain == roi'
print(f"{'级':<6s}{'gain':>10s}{'roi':>10s}")
for i, (g_, r_) in enumerate(zip(pl['gains'], pl['roi'])):
    print(f'{i:<6d}{g_:>10.4f}{r_:>10.4f}')
print('✅ 练习 1 通过：**排序会因为「除以成本」而反转** —— 这正是决策的价值所在。')

## ✏️ 练习 2：自适应 padding 的 crop 契约

实现 `adaptive_crop(box, min_pad=0.15, min_pad_px=3.0)`：

1. **正方形化**：以框中心为心，边长取 `max(w, h)`
2. **自适应 padding**：`pad = max(min_pad, min_pad_px / side)`（小框需要更大的相对余量）
3. 返回 `(x0, y0, x1, y1, pad)`，均为 float，不做边界裁剪

这就是两级方案的 API 契约 —— **一旦定下就不能随便改**。

In [ ]:
def adaptive_crop(box, min_pad=0.15, min_pad_px=3.0):
    # TODO: box = (x, y, w, h) -> 正方形化 -> 自适应 padding -> (x0, y0, x1, y1, pad)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
x0, y0, x1, y1, pad = adaptive_crop((100.0, 100.0, 60.0, 40.0))
assert abs(pad - 0.15) < 1e-9, '大框用 min_pad'
assert abs((x0 + x1) / 2 - 130.0) < 1e-9 and abs((y0 + y1) / 2 - 120.0) < 1e-9, '中心不变'
assert abs((x1 - x0) - (y1 - y0)) < 1e-9, '必须是正方形'
assert abs((x1 - x0) - 60.0 * 1.30) < 1e-9, '边长 = side × (1 + 2·pad)'

_, _, _, _, pad_small = adaptive_crop((10.0, 10.0, 12.0, 12.0))
assert abs(pad_small - 0.25) < 1e-9, '12 px 的框：3/12 = 0.25 > 0.15'
assert pad_small > pad, '**小框必须得到更大的相对 padding**'

_, _, _, _, pad_tiny = adaptive_crop((0.0, 0.0, 8.0, 8.0))
assert abs(pad_tiny - 0.375) < 1e-9
print(f"{'原始框 (w×h)':<16s}{'边长':>8s}{'pad':>8s}{'crop 边长':>12s}")
for b in [(0, 0, 8, 8), (0, 0, 12, 12), (0, 0, 32, 32), (0, 0, 60, 40)]:
    a, b_, c_, d_, p = adaptive_crop(tuple(float(v) for v in b))
    print(f'{f"{b[2]}×{b[3]}":<16s}{max(b[2], b[3]):>8d}{p:>8.3f}{c_ - a:>12.2f}')
print('✅ 练习 2 通过：**小框的检测误差相对量级更大 -> 需要更大的 padding 比例。**')

## ✏️ 练习 3：高置信度区的标定审计

单个 ECE 数字会让「低分区偏保守」与「高分区过自信」**互相抵消**。
而下游只会使用高置信度的检测 —— 所以必须单独审计高分区。

实现 `calibration_audit(conf, correct, n_bins=10, hi=0.8)`，返回：
- `'ece'`：整体 ECE（可直接调用上面已实现的 `ece`）
- `'hi_gap'`：`conf >= hi` 的样本上 `平均置信度 − 实际准确率`（**正数 = 过度自信**）
- `'hi_n'`：该区间样本数
- `'worst_bin'`：`|acc − conf|` 最大且样本数 ≥ 30 的桶的 `(lo, hi, gap)`

In [ ]:
def calibration_audit(conf, correct, n_bins=10, hi=0.8):
    # TODO: 返回 {'ece':…, 'hi_gap':…, 'hi_n':…, 'worst_bin': (lo, hi, gap)}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
au_raw = calibration_audit(conf_raw[tst], correct[tst])
au_cal = calibration_audit(conf_cal[tst], correct[tst])
assert abs(au_raw['ece'] - ece(conf_raw[tst], correct[tst])) < 1e-12
assert au_raw['hi_gap'] > 0.05, '未标定：高置信度区严重过度自信'
assert abs(au_cal['hi_gap']) < au_raw['hi_gap'] / 2, '标定后高置信度区的缺口大幅收窄'
assert au_raw['hi_n'] > 1000 and au_cal['hi_n'] > 500

# 构造一个「整体 ECE 很小、但高分区很糟」的例子 —— 说明为什么必须分区看
g = np.random.default_rng(5)
c_lo = np.full(6000, 0.30); y_lo = (g.random(6000) < 0.42).astype(float)   # 低分区偏保守
c_hi = np.full(6000, 0.95); y_hi = (g.random(6000) < 0.83).astype(float)   # 高分区过自信
cc = np.concatenate([c_lo, c_hi]); yy = np.concatenate([y_lo, y_hi])
au = calibration_audit(cc, yy)
assert au['hi_gap'] > 0.10, '高分区缺口 ≈ 0.95 − 0.83 = 0.12'
assert au['worst_bin'][2] > 0.10
print(f"整体 ECE = {au['ece']:.4f}   高分区(≥0.8) gap = {au['hi_gap']:+.4f}"
      f"   最差桶 = [{au['worst_bin'][0]:.1f},{au['worst_bin'][1]:.1f}) gap={au['worst_bin'][2]:.3f}")
print('✅ 练习 3 通过：**别只报一个 ECE 数字，必须看高置信度那几个桶。**')

## ✏️ 练习 4：标注预算怎么在两级之间分配

一个几乎每个团队都在拍脑袋决定的真问题：**有 B 条标注预算，
多少给「检测数据」、多少给「分类数据」？**

实现 `allocate_budget(budget, r_fn, a_fn, step=100)`：网格搜索所有划分，
返回 `{'n_det':…, 'n_cls':…, 'recall':…}`，使 `r_fn(n_det) * a_fn(n_cls)` 最大。

In [ ]:
def allocate_budget(budget, r_fn, a_fn, step=100):
    # TODO: 对 n_det 从 0 到 budget 按 step 网格搜索，最大化 r_fn(n_det) * a_fn(budget - n_det)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
R_of = lambda n: 0.99 * (1 - np.exp(-n / 2000.0))     # 检测：样本效率低（n0 大）
A_of = lambda n: 0.99 * (1 - np.exp(-n / 800.0))      # 分类：任务简单（n0 小），更快饱和

al = allocate_budget(6000, R_of, A_of, step=100)
assert al['n_det'] + al['n_cls'] == 6000
assert 3000 < al['n_det'] < 5000, f"最优解应在内部，得到 {al['n_det']}"
assert al['n_det'] > al['n_cls'], '饱和更慢的一级（检测）应该分到更多预算'
assert al['recall'] > R_of(3000) * A_of(3000), '最优划分优于 50/50 均分'
assert al['recall'] > R_of(6000) * A_of(0), '全给一级 = 另一级为 0 = 端到端为 0'

print(f"{'n_det':>8s}{'n_cls':>8s}{'R_det':>9s}{'A_cls':>9s}{'端到端':>9s}")
for nd in [1000, 3000, al['n_det'], 5000]:
    nc = 6000 - nd
    tag = '   <- 最优' if nd == al['n_det'] else ''
    print(f'{nd:>8d}{nc:>8d}{R_of(nd):>9.4f}{A_of(nc):>9.4f}{R_of(nd) * A_of(nc):>9.4f}{tag}')
print('✅ 练习 4 通过：**最优解总在内部** —— 因为乘积里任何一项为 0 都会毁掉全部。')
print('   这也是「短板优先」的严格版本：短板由 1−p 与该级的**边际斜率**共同决定。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def improvement_plan(rates, ceilings, costs):
    base = cascade_recall(rates)
    gains = []
    for i, c in enumerate(ceilings):
        r2 = list(rates); r2[i] = c
        gains.append(cascade_recall(r2) - base)
    roi = [g / c for g, c in zip(gains, costs)]
    return {'base': base, 'gains': gains, 'roi': roi,
            'best_gain': int(np.argmax(gains)), 'best_roi': int(np.argmax(roi))}

In [ ]:
# 练习 2 参考答案
def adaptive_crop(box, min_pad=0.15, min_pad_px=3.0):
    x, y, w, h = box
    side = max(w, h)
    cx, cy = x + w / 2.0, y + h / 2.0
    pad = max(min_pad, min_pad_px / side)          # 小框 -> 更大的相对余量
    half = side * (1.0 + 2.0 * pad) / 2.0
    return (cx - half, cy - half, cx + half, cy + half, pad)

In [ ]:
# 练习 3 参考答案
def calibration_audit(conf, correct, n_bins=10, hi=0.8):
    conf = np.asarray(conf, float); correct = np.asarray(correct, float)
    e, rows = ece(conf, correct, n_bins, return_bins=True)
    m = conf >= hi
    hi_gap = float(conf[m].mean() - correct[m].mean()) if m.sum() else float('nan')
    worst = (float('nan'), float('nan'), -1.0)
    for lo, hi_, n_, cf, acc in rows:
        if n_ >= 30 and abs(acc - cf) > worst[2]:
            worst = (lo, hi_, abs(acc - cf))
    return {'ece': e, 'hi_gap': hi_gap, 'hi_n': int(m.sum()), 'worst_bin': worst}

In [ ]:
# 练习 4 参考答案
def allocate_budget(budget, r_fn, a_fn, step=100):
    best = {'n_det': 0, 'n_cls': budget, 'recall': -1.0}
    for n_det in range(0, budget + 1, step):
        n_cls = budget - n_det
        r = float(r_fn(n_det) * a_fn(n_cls))
        if r > best['recall']:
            best = {'n_det': n_det, 'n_cls': n_cls, 'recall': r}
    return best

---
## 🧪 真实工程胶囊：两级 TSR 的接口契约 + 标定 + 日志 schema

下面这段可以原样复制进真实项目。三样东西缺一不可：
**冻结的 crop 契约**、**分场景的标定表**、**能归因到具体一级的日志**。

In [ ]:
RECIPE = r'''
# ─────────────────────────────────────────────────────────────
# ① crop 契约（configs/tsr_crop_contract.yaml）—— 改它 = 破坏性变更，两个模型都要重训
# ─────────────────────────────────────────────────────────────
crop_contract:
  version: "v3"                  # 每次变更 +1，模型权重文件名里带上它
  squarify: true                 # 以框中心扩成正方形（**保长宽比：形状是 TSR 的一级语义**）
  pad_ratio_min: 0.15            # 基础 padding
  pad_ratio_min_px: 3.0          # 自适应：pad = max(0.15, 3px / side)
  out_size: [64, 64]
  interp: bilinear               # 训练与车端必须**同一种插值**（见 C60 模块 01）
  border_mode: replicate         # 越界时的填充方式，也必须一致
  # 训练时对 GT 框施加的扰动 —— 必须与检测器的真实误差分布一致
  train_jitter: {center: 0.12, scale: 0.15}   # 从检测器验证集统计出来，不是拍脑袋

# 检测器输出的统计脚本（用来标定上面的 train_jitter）：
#   for gt, pred in matched_pairs(det_val_set):
#       dcx.append((pred.cx - gt.cx) / gt.w); dsc.append(pred.w / gt.w - 1)
#   train_jitter.center = np.percentile(np.abs(dcx), 90)
#   train_jitter.scale  = np.percentile(np.abs(dsc), 90)

# ─────────────────────────────────────────────────────────────
# ② 分场景 temperature scaling（纯后处理，不改排序 -> 不动 mAP）
# ─────────────────────────────────────────────────────────────
# 在**独立验证集**上按场景各拟合一个 T；线上按当前场景标签查表
CALIB = {
    "day_clear":  {"T_det": 1.42, "T_cls": 1.86},
    "night":      {"T_det": 1.95, "T_cls": 2.41},   # 夜间更过度自信
    "rain_fog":   {"T_det": 2.10, "T_cls": 2.28},
    "tunnel":     {"T_det": 2.35, "T_cls": 2.55},
}
def calibrated_score(s, T):
    z = np.log(np.clip(s, 1e-6, 1 - 1e-6) / (1 - np.clip(s, 1e-6, 1 - 1e-6)))
    return 1.0 / (1.0 + np.exp(-z / T))

p = calibrated_score(s_det, CALIB[scene]["T_det"]) * calibrated_score(s_cls, CALIB[scene]["T_cls"])

# ─────────────────────────────────────────────────────────────
# ③ 逐类代价阈值（拒识）：tau = c_FP / (c_FP + c_FN)
# ─────────────────────────────────────────────────────────────
COST = {"stop": (2, 200), "speed_limit": (8, 40), "no_uturn": (15, 25), "poi": (25, 1)}
TAU  = {k: cfp / (cfp + cfn) for k, (cfp, cfn) in COST.items()}
emit = p > TAU[cls_name]          # 上报 iff 期望代价更低

# ─────────────────────────────────────────────────────────────
# ④ 日志 schema —— **没有它，两级的可调试性优势直接归零**
# ─────────────────────────────────────────────────────────────
LOG_FIELDS = [
    "frame_id", "ts_ns", "camera_id",
    "det_box_xywh", "s_det_raw", "s_det_cal",     # 第一级：框 + 原始/标定分数
    "crop_box_xywh", "crop_pad", "contract_ver",  # 接口：实际用了什么 crop
    "cls_top3", "cls_top3_scores", "s_cls_cal",   # 第二级：top-3 而不只是 top-1
    "p_joint", "tau_used", "emitted", "reject_reason",
    "track_id", "n_frames_confirmed", "scene_tag",
]
# 体积：每帧每候选约 200–300 B。**这份日志是整个数据闭环的地基**
# （模块 05 的分桶评测、C58 的难例触发，全部建立在它上面）。

# ─────────────────────────────────────────────────────────────
# ⑤ 评测铁律（写进 CI）
# ─────────────────────────────────────────────────────────────
# 1) 分类器的验证集**必须用检测器输出的框**裁剪，不是 GT 框
# 2) 对外只报端到端 recall = R_det × A_cls，不报分级指标
# 3) 逐类 macro + 分尺寸分桶，micro 会淹没尾部类
# 4) 延迟报 p99 不报均值；crop 数固定 batch，方差比均值更重要
'''
print(RECIPE)
for key in ['crop_contract', 'train_jitter', 'calibrated_score', 'TAU', 'LOG_FIELDS',
            'cls_top3', 'contract_ver', 'p99']:
    assert key in RECIPE, key
print('✅ 配方覆盖：crop 契约 / 扰动统计 / 分场景标定 / 代价阈值 / 可归因日志 / 评测铁律')

### 小结

- **级联召回 = 检测召回 × 分类准确率**。对数域上各级损失相加，**没有任何一级能补偿另一级**。
  0.90 × 0.90 = 0.81 —— 两个「还行」的模块合起来是「不能用」。**对外只能汇报端到端。**
- **边际收益 ΔR = Δp_i × (其余各级之积)**。两级时两者往往接近 ⇒ 光看收益无法决策，
  必须引入**成本**与**天花板**；车端还要再乘一条**延迟代价**。
- **两级的收益几乎全部来自尾部类**（头部类谁都学得会）⇒ 评测必须看 **macro 与逐类**，
  micro 会把这个收益完全淹没。
- **crop 是两级方案的 API**。用 GT 框训练、用检测框推理会凭空损失十几个点，而**离线评测看不出来**。
  修法：训练时按检测器的真实误差分布扰动 GT 框；**验证集也必须用检测框裁**。
- **padding 要自适应**：`pad = max(0.15, 3px / side)`。小框的相对框误差更大。
  正方形化时**必须保长宽比**——形状本身是 TSR 的一级语义。
- **乘法只在两个分数都标定后才有概率语义**。temperature scaling 单参数、不改排序、纯后处理，
  几乎是白拿的改进；但**不能在训练集上标定**，且**必须分场景标定**。
- **别只报一个 ECE**：它会让低分区的保守与高分区的自信互相抵消。**看高置信度那几个桶。**
- **拒识阈值 τ = c_FP / (c_FP + c_FN)**，逐类不同。这要求 p 是标定过的概率 ——
  标定不是锦上添花，是代价敏感决策的前置条件。
- **级联的阈值应「前松后紧」**：早期漏检不可恢复（乘法里的 0 吞掉一切），
  早期误检可被后续更强的判别力低成本滤除。这条原则可迁移到任何级联系统。

下一站：**模块 03 · 失效模式全景** —— 把「哪里会错」拆成四象限并逐一深挖，
这是 JD 里「Analyze TSR-related scenarios and failure cases」的直接对应。